In [1]:
7

7

In [2]:
# identify.py

import os
import cv2
import numpy as np
import pickle
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from scipy.spatial.distance import cosine

# ------------------------------
# Load Pretrained Model (Face Embedding Extractor)
# ------------------------------
model = MobileNetV2(weights="imagenet", include_top=False, pooling='avg', input_shape=(224, 224, 3))

# ------------------------------
# Function: Get Face Embedding
# ------------------------------
def get_embedding(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    embedding = model.predict(img_array)[0]
    embedding = embedding / np.linalg.norm(embedding)  # normalize vector
    return embedding

# ------------------------------
# Step 1: Create Embeddings Database
# ------------------------------
def create_embeddings_database(train_path="img_dataset/train/"):
    embeddings_db = {}
    for person_name in os.listdir(train_path):
        person_dir = os.path.join(train_path, person_name)
        if not os.path.isdir(person_dir):
            continue
        
        person_embeddings = []
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            emb = get_embedding(img_path)
            person_embeddings.append(emb)
        
        # average embedding per person
        embeddings_db[person_name] = np.mean(person_embeddings, axis=0)
        print(f"✅ Saved embeddings for: {person_name}")
    
    with open("embeddings_db.pkl", "wb") as f:
        pickle.dump(embeddings_db, f)
    print("📁 Embeddings database created and saved!")

# ------------------------------
# Step 2: Identify a New Face
# ------------------------------
def identify_face(test_image_path):
    # load embeddings db
    with open("embeddings_db.pkl", "rb") as f:
        embeddings_db = pickle.load(f)

    test_emb = get_embedding(test_image_path)
    
    min_dist = 1
    identity = "Unknown"

    for name, db_emb in embeddings_db.items():
        dist = cosine(test_emb, db_emb)
        print(f"{name} → distance: {dist:.4f}")
        if dist < min_dist:
            min_dist = dist
            identity = name

    # threshold tuning
    threshold = 0.4
    if min_dist < threshold:
        print(f"\n✅ Identified as: {identity} (Distance={min_dist:.4f})")
    else:
        print("\n❌ Unknown Person")

# ------------------------------
# Run Program
# ------------------------------
if __name__ == "__main__":
    # Step 1: Create database once
    create_embeddings_database("img_dataset/train")
    
    # Step 2: Identify new face
    identify_face("img_dataset/test/16.jpg")

2025-10-26 20:07:34.883174: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-26 20:07:34.883405: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-26 20:07:34.921253: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
E0000 00:00:1761487657.472138   49891 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not b

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 666ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
✅ Saved embeddings for: fernandes
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
✅ Saved embeddings for: sinner
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━